In [ ]:
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes
from catboost import Pool
import pandas as pd
from datetime import datetime
import joblib

# Загрузка модели
model = joblib.load("XGBoost_model.pkl")


def prepare_features(date: str, prices: list) -> list:
    """Подготавливает данные для модели: дата + 6 числовых значений"""
    try:
        # Преобразуем дату в числовые признаки
        dt = datetime.strptime(date, "%d.%m.%Y")
        date_features = [
            float(dt.year),
            float(dt.month),
            float(dt.day),
            float(dt.isocalendar()[1])  # Номер недели
        ]

        # Проверяем цены
        if len(prices) != 6:
            raise ValueError("Нужно 6 цен")

        numeric_values = [float(x) for x in prices]

        # Объединяем дату и цены (порядок должен соответствовать model.feature_names_)
        return date_features + numeric_values

    except ValueError as e:
        raise ValueError(f"Ошибка формата: {str(e)}")


async def start(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    help_text = (
        "Введите данные в формате:\n"
        "ДД.ММ.ГГГГ,цена1,цена2,цена3,цена4,цена5,цена6\n\n"
        "Пример:\n"
        "25.03.2025,15000,15500,15200,16000,14800,450\n\n"
        "Где:\n"
        "1. Дата (ДД.ММ.ГГГГ)\n"
        "2. Лом_3А, Южный ФО (руб/т)\n"
        "3. Лом_3А, Центральный ФО (руб/т)\n"
        "4. Лом_3А, Татарстан (руб/т)\n"
        "5. Лом_3А, Московский регион (руб/т)\n"
        "6. Лом_3А, Уральский ФО (руб/т)\n"
        "7. Чугун, Турция ($/т)"
    )
    await update.message.reply_text(help_text)


async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    try:
        parts = update.message.text.split(",")
        if len(parts) != 7:
            raise ValueError("Нужно 7 значений через запятую: дата + 6 цен")

        date = parts[0].strip()
        prices = parts[1:7]

        # Подготавливаем данные
        features = prepare_features(date, prices)

        # Создаём Pool для CatBoost
        pool = Pool(
            data=[features],
            feature_names=list(model.feature_names_)
        )

        prediction = model.predict(pool)[0]
        await update.message.reply_text(f"📅 Дата: {date}\n📊 Прогноз: {prediction:.2f}")

    except Exception as e:
        await update.message.reply_text(f"❌ Ошибка: {str(e)}")


def main():
    application = Application.builder().token("Token").build()
    application.add_handler(CommandHandler("start", start))
    application.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    application.run_polling()


if __name__ == "__main__":
    main()